# Signups forecasting — US charts (`signups_forecasting_country.py`)

This notebook loads the pipeline module, fits **all** country + region/KR models (same as `main()`), then plots **forecast** and **components** for two US series:

| Your label | Prophet model id in this script |
|------------|----------------------------------|
| `M14_R3_US_EDU_MAR_WEB` | `M14_R3_US_EDU_MAR_WEB` |
| `M17_R3_US_B2C_ORG_WEB` | `M17_R3_US_NON_ORG_WEB` |

Non-edu organic web is encoded as **`NON_ORG_WEB`** in `signups_forecasting_country.py` (aligned with the R `signups_forecasting_country.r` split). Older US-only scripts may label the same series `B2C_ORG_WEB` in extracts — the fitted column name here is still `M17_R3_US_NON_ORG_WEB`.

**Setup:** open from repo root `profit-from-prophet`, or set `REPO` in the import cell. Use a kernel with `pandas`, `prophet`, `matplotlib`, `openpyxl` installed.


In [ ]:
%pip install -q pandas numpy prophet matplotlib openpyxl


In [ ]:
%matplotlib inline

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path(".").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import signups_forecasting_country as sfc


## Optional: override paths


In [ ]:
# sfc.INPUT_DIRECTORY = str(REPO / "path" / "to" / "Prophet Inputs")
# sfc.OUTPUT_DIRECTORY = str(REPO / "path" / "to" / "Prophet Outputs")

print("IN:", sfc.INPUT_DIRECTORY)
print("OUT:", sfc.OUTPUT_DIRECTORY)


## Load data and fit all models


In [ ]:
os.makedirs(sfc.OUTPUT_DIRECTORY, exist_ok=True)

model_names, _a, _f = sfc.build_model_names()
data = sfc.load_and_prepare_data()
state = sfc.RunState()

print(
    f"Actuals to: {sfc.D_MS_ATOF.date()} | Forecast: {sfc.D_FORECAST_START.date()} → {sfc.D_FORECAST_END.date()}"
)
print(f"Models: {len(model_names)} | Signups rows: {len(data.su):,}")

fc_time, fr_time = sfc.run_country_and_region_forecasts(state, data)
print(f"Fit done — Country: {fc_time:.1f}s | Region/KR: {fr_time:.1f}s")


## Charts: `M14_R3_US_EDU_MAR_WEB` and M17 US “B2C org web” (`M17_R3_US_NON_ORG_WEB`)

Four figures per run: main forecast, then seasonality/trend **components** for each model.


In [ ]:
# Prophet keys in this pipeline (M17 B2C organic web = NON_ORG_WEB suffix)
INSPECT_MODELS = (
    "M14_R3_US_EDU_MAR_WEB",
    "M17_R3_US_NON_ORG_WEB",
)
CHART_TITLES = {
    "M14_R3_US_EDU_MAR_WEB": "M14_R3_US_EDU_MAR_WEB",
    "M17_R3_US_NON_ORG_WEB": "M17_R3_US_B2C_ORG_WEB (same fit as M17_R3_US_NON_ORG_WEB)",
}

for name in INSPECT_MODELS:
    if name not in state.models:
        print(f"Missing model {name!r} — check fits completed.")
        continue
    fc = state.forecasts[f"{name}_F"]
    m = state.models[name]
    label = CHART_TITLES.get(name, name)

    fig = m.plot(fc)
    fig.suptitle(f"Forecast — {label}")
    if fig.axes:
        fig.axes[0].set_xlabel("Year")
        fig.axes[0].set_ylabel("Sign Ups")
    plt.tight_layout()
    plt.show()

    fig_c = m.plot_components(fc)
    fig_c.suptitle(f"Components — {label}")
    plt.tight_layout()
    plt.show()
